# Resiliência de Ativos Brasileiros em Crises — Post 1

**Projeto público:** análise quantitativa de como diferentes classes de ativos brasileiros
se comportaram em três episódios de stress relevantes.

---

## Definições operacionais

### Resiliência (definição adotada neste projeto)
Capacidade de um ativo de **preservar valor** (drawdown baixo) **E** de **recuperar ao nível
pré-evento em prazo razoável** (tempo de recuperação curto). Nenhuma dessas dimensões,
isolada, caracteriza resiliência completa.

### Janelas de análise
| Janela | Definição | Propósito |
|--------|-----------|----------|
| **Curta** | T−5 a T+90 dias úteis | Impacto imediato e recuperação inicial |
| **Longa** | T−5 a T+180 dias úteis | Consolidação da recuperação |

Ambas as janelas são calculadas. A comparação entre elas sinaliza recuperações incompletas
ou reversões tardias.

### Métricas calculadas
1. **Drawdown máximo** — maior queda pico-a-vale dentro da janela (%).
2. **Tempo de recuperação** — dias úteis do pico de drawdown ao retorno ao nível T−5.
3. **Drawdown normalizado** — drawdown_max / vol_pré_evento (razão de intensidade relativa).
4. **Retorno acumulado da janela** — variação total do ativo no período completo.

### CDI como referência, não como ativo comparável
O CDI é a taxa básica do mercado interbancário brasileiro — o custo de oportunidade
de não estar exposto a risco de mercado. Ele não sofre drawdowns relevantes por
construção, portanto **não é comparável** com fundos multimercado ou índices de
renda variável. Nos gráficos, aparece como linha tracejada de referência.

---

> ⚠️ **Linguagem cautelosa:** este notebook observa e descreve padrões quantitativos.
> Não atribui causalidade entre os eventos e os movimentos de preço observados.
> Expressões como "coincidiu com" e "observou-se" são usadas intencionalmente.


## Eventos analisados e justificativa das datas

### Covid-19 — 21/02/2020
Sexta-feira marcada por forte deterioração dos mercados globais após a explosão de casos
na Itália e no Irã. Considera-se essa data o primeiro pico real de stress sistêmico
internacional — quando o mercado processou que o vírus havia ultrapassado a contenção
chinesa. Semanas anteriores já apresentavam sinais de preocupação, mas 21/02 é o ponto
de ruptura amplamente reconhecido em análises ex-post.

### Invasão da Ucrânia — 24/02/2022
Data de início da invasão militar russa ao território ucraniano. Sem ambiguidade como
ponto de materialização do risco geopolítico — a partir dessa data o choque de
commodities e a reação dos Bancos Centrais globais tornaram-se centrais para
a precificação de ativos.

### Americanas — 11/01/2023
Data de publicação do fato relevante da Americanas S.A. comunicando inconsistências
contábeis de R$ 20 bilhões. Evento de crédito privado com impacto imediato sobre
fundos multimercado e fundos de crédito com exposição ao papel. A data é precisa
e sem ambiguidade na literatura de mercado.

---

> **Nota metodológica:** o uso de T−5 como início da janela (em vez de T+0) captura
> eventuais movimentos antecipados nos dias imediatamente anteriores ao evento formal,
> mas mantém T+0 como referência visual nos gráficos.


In [ ]:
import sys
import logging
import warnings
from pathlib import Path

# Adicionar raiz do projeto ao path para importar src/
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s | %(name)s | %(message)s')
logger = logging.getLogger('post1')

%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

from src.data_fetch.bcb_fetcher    import fetch_cdi, build_cdi_index
from src.data_fetch.yfinance_fetcher import fetch_ibovespa
from src.data_fetch.anbima_fetcher  import fetch_ifmm, fetch_ima_b5, AnbimaDataUnavailableError
from src.metrics.resilience_metrics import analyze_event
from src.viz.post1_charts import (
    plot_normalized_series,
    plot_drawdown_heatmap,
    plot_recovery_bars,
    plot_summary_table,
)

FIGURES_DIR = PROJECT_ROOT / 'figures' / 'post1'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Setup concluído. PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
# ── Definição dos eventos ──────────────────────────────────────────────────
EVENTS = {
    'Covid': {
        'date': pd.Timestamp('2020-02-21'),
        'label': 'Covid-19 (21/02/2020)',
        'justificativa': (
            'Sexta-feira de ruptura global: explosão de casos na Itália e Irã. '
            'Primeiro pico real de stress sistêmico internacional.'
        ),
    },
    'Ukraine': {
        'date': pd.Timestamp('2022-02-24'),
        'label': 'Invasão da Ucrânia (24/02/2022)',
        'justificativa': (
            'Data de início da invasão militar russa. '
            'Sem ambiguidade como ponto de materialização do risco geopolítico.'
        ),
    },
    'Americanas': {
        'date': pd.Timestamp('2023-01-11'),
        'label': 'Americanas (11/01/2023)',
        'justificativa': (
            'Fato relevante de inconsistências contábeis de ~R$20bi. '
            'Evento de crédito privado com impacto em fundos com exposição ao papel.'
        ),
    },
}

WINDOWS  = ['90d', '180d']
ASSETS   = ['ibov', 'ifmm', 'ima_b5']  # CDI é referência, tratado separadamente

# Período de download: cobre todos os eventos + 180du antes/depois com margem
DOWNLOAD_START = '2019-06-01'
DOWNLOAD_END   = '2023-12-31'
DOWNLOAD_START_BCB = '01/06/2019'
DOWNLOAD_END_BCB   = '31/12/2023'

print('Eventos configurados:')
for name, info in EVENTS.items():
    t5 = info['date'] - pd.tseries.offsets.BusinessDay(5)
    t90 = info['date'] + pd.tseries.offsets.BusinessDay(90)
    t180 = info['date'] + pd.tseries.offsets.BusinessDay(180)
    print(f"  {name}: T-5={t5.date()} | T+0={info['date'].date()} "
          f"| T+90={t90.date()} | T+180={t180.date()}")


In [ ]:
# ── Download: Ibovespa ────────────────────────────────────────────────────
print('Buscando Ibovespa (^BVSP via yfinance)...')
ibov = fetch_ibovespa(DOWNLOAD_START, DOWNLOAD_END)
print(f'  OK | {len(ibov)} obs | {ibov.index[0].date()} a {ibov.index[-1].date()}')
print(f'  Últimos 3 fechamentos:')
display(ibov.tail(3).rename('Ibovespa (pts)'))


In [ ]:
# ── Download: CDI ─────────────────────────────────────────────────────────
print('Buscando taxa CDI (BCB SGS série 12)...')
cdi_rates = fetch_cdi(DOWNLOAD_START_BCB, DOWNLOAD_END_BCB)
print(f'  OK | {len(cdi_rates)} obs | {cdi_rates.index[0].date()} a {cdi_rates.index[-1].date()}')
print(f'  Taxa CDI média no período: {cdi_rates.mean():.4f}% ao dia '
      f'({((1 + cdi_rates.mean()/100)**252 - 1)*100:.1f}% a.a. aproximado)')
display(cdi_rates.tail(3).rename('CDI (% ao dia)'))


In [ ]:
# ── Download: IFMM ────────────────────────────────────────────────────────
print('Buscando IFMM (ANBIMA)...')
ifmm = None
try:
    ifmm = fetch_ifmm(DOWNLOAD_START, DOWNLOAD_END)
    print(f'  OK | {len(ifmm)} obs | {ifmm.index[0].date()} a {ifmm.index[-1].date()}')
    display(ifmm.tail(3).rename('IFMM'))
except AnbimaDataUnavailableError as e:
    print(f'  ⚠ IFMM indisponível: {e}')
    print('  → Siga as instruções em src/data_fetch/anbima_fetcher.py para download manual.')
except Exception as e:
    print(f'  ⚠ Erro inesperado ao buscar IFMM: {type(e).__name__}: {e}')


In [ ]:
# ── Download: IMA-B 5 ────────────────────────────────────────────────────
print('Buscando IMA-B 5 (ANBIMA)...')
ima_b5 = None
try:
    ima_b5 = fetch_ima_b5(DOWNLOAD_START, DOWNLOAD_END)
    print(f'  OK | {len(ima_b5)} obs | {ima_b5.index[0].date()} a {ima_b5.index[-1].date()}')
    display(ima_b5.tail(3).rename('IMA-B 5'))
except AnbimaDataUnavailableError as e:
    print(f'  ⚠ IMA-B 5 indisponível: {e}')
    print('  → Siga as instruções em src/data_fetch/anbima_fetcher.py para download manual.')
except Exception as e:
    print(f'  ⚠ Erro inesperado ao buscar IMA-B 5: {type(e).__name__}: {e}')


In [ ]:
# ── Sanity checks ─────────────────────────────────────────────────────────
print('=== SANITY CHECKS ===' + '='*40)

series_available = {'ibov': ibov, 'ifmm': ifmm, 'ima_b5': ima_b5}

for name, s in series_available.items():
    if s is None:
        print(f'  [{name}] AUSENTE — métricas serão NaN')
        continue
    n_na = s.isna().sum()
    pct_na = 100 * n_na / len(s)
    n_zeros = (s == 0).sum()
    print(f'  [{name}] {len(s)} obs | NaN: {n_na} ({pct_na:.1f}%) | zeros: {n_zeros}')
    print(f'          min={s.min():.2f} | max={s.max():.2f} | último={s.iloc[-1]:.2f}')

    # Verificar cobertura de cada evento
    for ev_name, ev_info in EVENTS.items():
        t_ev = ev_info['date']
        t5  = t_ev - pd.tseries.offsets.BusinessDay(5)
        t180 = t_ev + pd.tseries.offsets.BusinessDay(180)
        n_in_window = s.loc[t5:t180].dropna()
        cobertura = f'{len(n_in_window)} obs de ~185 esperados'
        status = '✓' if len(n_in_window) > 100 else '⚠ cobertura parcial'
        print(f'          {ev_name}: {status} ({cobertura})')

# CDI
print(f'  [cdi] {len(cdi_rates)} obs | '
      f'min={cdi_rates.min():.4f}% | max={cdi_rates.max():.4f}% ao dia')


In [ ]:
# ── Cálculo de métricas: todos os ativos × eventos × janelas ──────────────
print('Calculando métricas...')

# Mapear CDI como série de índice acumulado para análise (referência)
# Nota: CDI não é incluído nos _ORDERED_ASSETS para fins de comparação direta.
# Calculamos as métricas do CDI apenas para contexto na tabela.

all_results = []

series_map = {
    'ibov':   ibov,
    'ifmm':   ifmm,
    'ima_b5': ima_b5,
}

# Adicionar CDI como ativo de referência (índice acumulado)
# O CDI índice começa em 100 na data mais antiga disponível
if cdi_rates is not None and len(cdi_rates) > 0:
    cdi_index = build_cdi_index(cdi_rates, cdi_rates.index[0])
    series_map['cdi'] = cdi_index
else:
    series_map['cdi'] = None

for ativo, serie in series_map.items():
    if serie is None:
        print(f'  [{ativo}] Skipped (sem dados)')
        continue
    for ev_name, ev_info in EVENTS.items():
        for janela in WINDOWS:
            result = analyze_event(
                serie=serie,
                ativo_nome=ativo,
                t_event=ev_info['date'],
                janela=janela,
            )
            result['evento'] = ev_name  # chave legível
            all_results.append(result)

print(f'  {len(all_results)} combinações calculadas.')


In [ ]:
# ── DataFrame consolidado ─────────────────────────────────────────────────
df = pd.DataFrame(all_results)

# Ordem de apresentação
col_order = [
    'ativo', 'evento', 'janela',
    'drawdown_max', 'data_drawdown_max', 'retorno_acumulado_janela',
    'vol_pre_evento', 'drawdown_normalizado',
    'recuperou', 'dias_recuperacao', 'data_recuperacao',
    't_start', 't_end',
]
df = df[[c for c in col_order if c in df.columns]]

# Formatar para exibição
df_display = df.copy()
for col in ['drawdown_max', 'retorno_acumulado_janela', 'vol_pre_evento']:
    if col in df_display:
        df_display[col] = df_display[col].map(lambda x: f'{x*100:.2f}%' if pd.notna(x) else '—')
if 'drawdown_normalizado' in df_display:
    df_display['drawdown_normalizado'] = df_display['drawdown_normalizado'].map(
        lambda x: f'{x:.2f}σ' if pd.notna(x) else '—'
    )

display(df_display.set_index(['ativo', 'evento', 'janela']))

# Salvar versão numérica para uso posterior
df_numeric = df.copy()


In [ ]:
# ── Visualização 1: Séries normalizadas ───────────────────────────────────
fig1 = plot_normalized_series(
    series_dict=series_map,
    events=EVENTS,
    cdi_rates=cdi_rates,
    output_path=FIGURES_DIR / 'fig1_series_normalizadas.png',
)
plt.show()


In [ ]:
# ── Visualização 2: Heatmap de drawdown ────────────────────────────────────
fig2 = plot_drawdown_heatmap(
    results_df=df_numeric,
    output_path=FIGURES_DIR / 'fig2_heatmap_drawdown.png',
)
plt.show()


In [ ]:
# ── Visualização 3: Tempo de recuperação ──────────────────────────────────
fig3 = plot_recovery_bars(
    results_df=df_numeric,
    janela='180d',
    output_path=FIGURES_DIR / 'fig3_recovery_bars.png',
)
plt.show()


In [ ]:
# ── Visualização 4: Tabela consolidada ────────────────────────────────────
fig4 = plot_summary_table(
    results_df=df_numeric,
    output_path=FIGURES_DIR / 'fig4_tabela_consolidada.png',
)
plt.show()


In [ ]:
# ── Observações automáticas ───────────────────────────────────────────────
# Esta seção gera estatísticas descritivas para auxiliar a redação do post.
# NÃO publica conclusões — apenas organiza os números observados.

print('=' * 65)
print('OBSERVAÇÕES AUTOMÁTICAS — subsídios para redação do post')
print('=' * 65)

df_num = df_numeric.copy()

# 1. Maior e menor drawdown observado (janela 90d)
df_90 = df_num[(df_num['janela'] == '90d') & (df_num['ativo'] != 'cdi')]
if not df_90.empty and df_90['drawdown_max'].notna().any():
    idx_min = df_90['drawdown_max'].idxmin()
    idx_max = df_90['drawdown_max'].idxmax()
    row_min = df_90.loc[idx_min]
    row_max = df_90.loc[idx_max]
    print(f'\n[1] Maior drawdown (janela 90d):')
    print(f'    {row_min["ativo"]} no evento {row_min["evento"]}: '
          f'{row_min["drawdown_max"]*100:.1f}%')
    print(f'    (ocorreu em: {row_min["data_drawdown_max"]}')
    print(f'\n    Menor drawdown (janela 90d):')
    print(f'    {row_max["ativo"]} no evento {row_max["evento"]}: '
          f'{row_max["drawdown_max"]*100:.1f}%')

# 2. Ativo com maior diferença entre janela curta e longa
df_90_vals  = df_num[df_num['janela'] == '90d'].set_index(['ativo', 'evento'])['drawdown_max']
df_180_vals = df_num[df_num['janela'] == '180d'].set_index(['ativo', 'evento'])['drawdown_max']
diff = (df_90_vals - df_180_vals).dropna()
if not diff.empty:
    idx_max_diff = diff.abs().idxmax()
    print(f'\n[2] Maior diferença DD entre janela 90d e 180d:')
    print(f'    {idx_max_diff}: {diff[idx_max_diff]*100:.1f}pp')
    print(f'    (sinaliza recuperação incompleta ou piora tardia nesse par)')

# 3. Recuperações parciais vs. completas (janela 180d)
df_180 = df_num[(df_num['janela'] == '180d') & (df_num['ativo'] != 'cdi')]
if not df_180.empty:
    rec_completas = df_180[df_180['recuperou'] == True]
    rec_parciais  = df_180[df_180['recuperou'] == False]
    print(f'\n[3] Recuperações na janela 180d:')
    print(f'    Completas ({len(rec_completas)} de {len(df_180)}):')
    for _, r in rec_completas.iterrows():
        dias_str = f'{r["dias_recuperacao"]} du' if pd.notna(r["dias_recuperacao"]) else '0 du'
        print(f'      ✓ {r["ativo"]} | {r["evento"]} | {dias_str}')
    print(f'    Incompletas ({len(rec_parciais)} de {len(df_180)}):')
    for _, r in rec_parciais.iterrows():
        print(f'      ✗ {r["ativo"]} | {r["evento"]}')

# 4. IFMM vs Ibov: em quantos eventos IFMM teve drawdown < 50% do Ibov
df_ibov  = df_90.set_index('evento')['drawdown_max'].rename('ibov')
df_ifmm  = df_num[(df_num['janela'] == '90d') & (df_num['ativo'] == 'ifmm')]
if not df_ifmm.empty and not df_ibov.empty:
    df_ifmm_idx = df_ifmm.set_index('evento')['drawdown_max']
    comparacoes = []
    for ev in df_ibov.index:
        if ev in df_ifmm_idx.index:
            dd_ibov = df_ibov[ev]
            dd_ifmm = df_ifmm_idx[ev]
            if pd.notna(dd_ibov) and pd.notna(dd_ifmm) and dd_ibov != 0:
                razao = dd_ifmm / dd_ibov
                comparacoes.append((ev, dd_ibov, dd_ifmm, razao))

    if comparacoes:
        print(f'\n[4] IFMM vs. Ibovespa — drawdown relativo (janela 90d):')
        n_metade = sum(1 for _, _, _, r in comparacoes if r < 0.5)
        for ev, dd_ibov, dd_ifmm, razao in comparacoes:
            flag = '< 50% do Ibov ✓' if razao < 0.5 else '≥ 50% do Ibov'
            print(f'    {ev}: Ibov={dd_ibov*100:.1f}% | IFMM={dd_ifmm*100:.1f}% '
                  f'| razão={razao:.2f} | {flag}')
        print(f'\n    IFMM com DD < metade do Ibov: '
              f'{n_metade} de {len(comparacoes)} eventos observados.')

# 5. Drawdown normalizado por volatilidade
df_nd = df_90[df_90['drawdown_normalizado'].notna()].sort_values('drawdown_normalizado')
if not df_nd.empty:
    print(f'\n[5] Drawdown normalizado (DD / vol anual pré-evento) — ranking janela 90d:')
    for _, r in df_nd.iterrows():
        print(f'    {r["ativo"]} | {r["evento"]}: {r["drawdown_normalizado"]:.2f}σ '
              f'(vol pré-evento: {r["vol_pre_evento"]*100:.1f}% a.a.)')

print('\n' + '=' * 65)
print('Fin. — use os números acima para redigir o post.')
print('=' * 65)


---

## Limitações conhecidas desta análise

1. **Dados ANBIMA (IFMM e IMA-B 5):** acesso depende de download manual caso o
   automatizado falhe. URLs sujeitas a mudança; verificar periodicamente.

2. **Ibovespa via yfinance:** pode apresentar ajustes retroativos não sinalizados.
   Para publicações definitivas, validar contra série oficial da B3.

3. **Dias úteis sem feriados:** `pd.bdate_range` usa calendário Mon-Fri puro;
   feriados brasileiros (Carnaval, Corpus Christi etc.) não são excluídos.
   A contagem de dias de recuperação pode diferir em ~1-3 dias úteis.

4. **Janela T-5 como base:** assume que o ativo não estava em queda antecipada.
   Em eventos com antecipação de mercado (ex: stress gradual pré-Covid),
   o drawdown medido pode estar subestimado.

5. **Drawdown normalizado:** a divisão de drawdown cumulativo por volatilidade
   anualizada não é um z-score estatístico estrito — é uma métrica de intensidade
   relativa. Comparação entre ativos com frequências diferentes (ex: IFMM diário vs.
   séries com gaps) deve ser feita com cautela.

6. **IFMM como proxy de multimercado:** representa a média ponderada por PL;
   fundos individuais podem divergir significativamente do índice.

7. **Escopo temporal:** três eventos entre 2020 e 2023 não constituem amostra
   suficiente para inferência estatística robusta.

---

## Roadmap dos próximos posts

- **Post 2:** Decomposição dos retornos — quanto da queda do IFMM veio de renda variável
  vs. crédito privado vs. exposição direcional?
- **Post 3:** Comparação com crise de 2018 (eleições) e 2015-16 (recessão).
- **Post 4:** Simulação de portfólio — como diferentes alocações entre Ibov e IMA-B 5
  teriam se comportado nos três eventos?
- **Post 5:** Análise de correlação dinâmica — os ativos ficaram mais correlacionados
  durante o stress?
- **Post 6:** Checklist de resiliência para avaliar novos fundos/índices.
